# Execution times for SV

## Imports and data loading

In [1]:
import os
import sys

# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["KERAS_BACKEND"] = "torch"
module_path = os.path.abspath(os.path.join('..' + os.sep + '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import pandas as pd
import scipy

In [3]:
CONST_ROUTE_MAIN_DIR = ".." + os.sep + ".." + os.sep

In [4]:
def get_times_from_route(aggregation_method, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario):
    if type_of_partition == "dirichlet":
        path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "times" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + "alpha_" + str(additional_parameter) + os.sep + "mlp" + os.sep + str(random_seed)
    else:
        if is_maverick_scenario:
            path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "times" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + additional_parameter+"_incl_maverick" + os.sep + "mlp" + os.sep + str(random_seed)
        else:
            path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "times" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + additional_parameter + os.sep + "mlp" + os.sep + str(random_seed)
    return path_to_result_dataframes

In [5]:
import itertools

def generate_experiments() -> list[dict]:
    # Valid (strategy, model) pairs — matched by shared type key (nn/dt)

    # strategies = {"nn": ["FedAvg"], "dt": ["FedXGB"]}
    # models = {"nn": ["mlp"], "dt": ["xgboost"]}
    strategies = {"nn": ["FedAvg", "FedProx", "Scaffold"]}
    # strategies = {"nn": ["FedAvg", "FedProx"]}
    # strategies = {"nn": ["FedAvg"]}
    # strategies = {"nn": ["FedProx"]}
    models = {"nn": ["mlp"]}
    datasets = {
        "classification": [
            # "wine",
            "har",
        #     # "adult",
        #     # "covertype",
        #     # "heart",
        #     # "binary_heart"
            "edge-iot-coreset"
        ],
        "regression": [
            "wids_dataset"
        ]
    }

    partition_types = [
        "dirichlet",
        # "feature_skew",
        "manual"
    ]

    hyperparameter_search_rounds = 30
    rounds = 50

    metrics = {"classification": ["CrossEntropyLoss", "Accuracy", "F1Score", "MCC", "F1ScoreMacro", "F1ScoreMicro"],
               "regression": ["MSE", "RMSE", "MAE"]}

    DEFAULT_DATA_SPLIT = [20, 20, 20, 20, 20]
    data_split = {
        "har": [16, 16, 16, 16, 16, 16],
    }

    alpha = [
        0.1,
        10,
        1000,
    ]
    n_clients = [5]
    run_number_list = [number for number in range(1, 6)]
    # run_number_list = [1]

    # Manual partitions come in two shapes:
    #  - a plain list of names (no maverick semantics)
    #  - a dict {name: (is_maverick, maverick_client_number, eval_type, special_class)}
    #    for scenarios where one client is a "maverick" holding unique data. I separate between Mavericks of class
    #    and feature by string.
    manual_partition_names = {
        "har": {
            "HAR_1_Maverick_Laying": (True, 0, "Class", 0, None),
            "HAR_1_Maverick_WalkingUpstairs": (True, 5, "Class", 5, None),
            "HAR_1_Maverick_Laying_Balanced": (True, 0, "Class", 0, None),
            "HAR_1_Maverick_WalkingUpstairs_Balanced": (True, 5, "Class", 5, None),
            # "HAR_1_Maverick_1_MissingOneLabel": (True, 5, "Class", 5, None),
            # "HAR_1_Maverick_1_MissingTwoLabels": (True, 5, "Class", 5, None),
        },
        "edge-iot-coreset": {
            "edgeiot_coreset_1_Maverick_Least_Class": (True, 4, "Class", 6, None),
            "edgeiot_coreset_1_Maverick_Only_Normal": (True, 4, "Class", 7, None),
            "edgeiot_coreset_1_Maverick_sql_injection": (True, 4, "Class", 11, None),
            "edgeiot_coreset_1_Maverick_ddos_udp": (True, 4, "Class", 4, None),
        },
        "wids_dataset": {
            "wids_energy_non_iid_by_label": (False, None, None, None, None),
            "wids_energy_iid_sampling": (False, None, None, None, None),
            "wids_energy_feature_skew_building_type": (False, None, None, None, None),
            "wids_energy_feature_skew_state_factor": (False, None, None, None, None),
            # "wids_energy_feature_skew_facility_type": (False, None, None, None, None),
            "wids_energy_maverick_facility_type_grocery_store": (
                True, 5, "Feature", ["facility_type"], "Grocery_store_or_food_market"
            ),
            "wids_energy_maverick_facility_type_uncategorized_multifamily": (
                True, 5, "Feature", ["facility_type"], "Multifamily_Uncategorized"
            ),
        }
    }

    strategy_model_pairs = [
        (strategy, model)
        for type_key in set(strategies) & set(models)
        for strategy in strategies[type_key]
        for model in models[type_key]
    ]

    dataset_metrics_pairs = [
        (dataset, metrics[type_key], type_key)
        for type_key in set(datasets) & set(metrics)
        for dataset in datasets[type_key]
    ]

    experiments = []

    for (strategy, model), (dataset, selected_metrics, task_type), number in itertools.product(
            strategy_model_pairs, dataset_metrics_pairs, run_number_list
    ):
        base = {
            "strategy": strategy,
            "model": model,
            "dataset": dataset,
            "metrics": selected_metrics,
            "run_number": number
        }

        # Regression tasks are restricted to manual partitions only.
        allowed_partitions = ["manual"] if task_type == "regression" else partition_types

        for partition in allowed_partitions:
            entry = {**base, "partition": partition}

            if partition == "dirichlet":
                split = data_split.get(dataset, DEFAULT_DATA_SPLIT)
                for a in alpha:
                    experiments.append({**entry, "alpha": a, "data_split": split})

            elif partition == "feature_skew":
                for nc in n_clients:
                    experiments.append({**entry, "n_clients": nc})

            elif partition == "manual":
                partitions = manual_partition_names.get(dataset, [])

                if isinstance(partitions, dict):
                    # Maverick-style entries — generate both a "with maverick"
                    # and a "without maverick" training run per scenario.
                    for name, (is_maverick, maverick_client_number, eval_type,
                               special_class_or_columns_for_subpopulation,
                               maverick_category_value) in partitions.items():
                        if is_maverick:
                            for exclude_maverick in (True, False):
                                experiments.append({
                                    **entry,
                                    "name": name,
                                    "is_maverick_scenario": True,
                                    "maverick_client_number": maverick_client_number,
                                    "maverick_eval_type": eval_type,
                                    "maverick_special_class_or_columns_for_subpopulation": special_class_or_columns_for_subpopulation,
                                    "maverick_category_value": maverick_category_value,
                                    "exclude_maverick": exclude_maverick,
                                })
                        else:
                            experiments.append({**entry, "name": name})
                else:
                    for name in partitions:
                        experiments.append({**entry, "name": name})

    return experiments

training_configurations_for_distances = generate_experiments()
# print(training_configurations_for_distances)

## Execution Times

In [6]:
def get_times_dataframe(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario):
    route = get_times_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario)
    # time_columns = ["BeforeEvaluation", "AfterEvaluation", "BeforeSV", "AfterSV", "BeforeGradientRewards", "AfterGradientRewards", "BeforeLocalTraining", "AfterLocalTraining", "BeforeLocalPartialEvaluations", "AfterLocalPartialEvaluations"]
    time_dataframe_list = []
    time_dataframe = pd.DataFrame()
    for file in os.listdir(route):
        if not os.path.isfile(os.path.join(route, file)):
            continue
        partial_dataframe = pd.read_csv(route + os.sep + file, index_col=0)
        if file == "server":
            start_of_evaluation = partial_dataframe.iloc[0, 0]
            evaluator = "Server"
            number_of_rounds = len(partial_dataframe)
        else:
            evaluator = "Client_" + file
        time_dataframe_list.append((evaluator, partial_dataframe))

    for evaluator, dataframe in time_dataframe_list:
        dataframe = dataframe - start_of_evaluation
        dataframe["Aggregation_Process"] = [aggregation_process for _ in range(len(dataframe))]
        dataframe["Dataset"] = [dataset_name for _ in range(len(dataframe))]
        dataframe["Type_of_partition"] = [type_of_partition for _ in range(len(dataframe))]
        dataframe["Dirichlet/NamePartition"] = [additional_parameter for _ in range(len(dataframe))]
        dataframe["Random_Seed"] = [random_seed for _ in range(len(dataframe))]
        dataframe["Round"] = dataframe.index
        dataframe["Evaluator"] = evaluator
        dataframe = dataframe.iloc[:number_of_rounds + 1]
        time_dataframe = pd.concat([time_dataframe, dataframe], ignore_index=True)

    return time_dataframe


In [7]:
# Load each timing directory once and retain an inventory of expected runs.
from pathlib import Path

EXPERIMENT_COLUMNS = ["Aggregation_Process", "Dataset", "Type_of_partition",
                      "Dirichlet/NamePartition", "Random_Seed"]
frames, expected_records, loading_records = [], [], []
seen_routes = set()
for configuration in training_configurations_for_distances:
    partition = configuration["partition"]
    parameter = configuration["alpha"] if partition == "dirichlet" else configuration["name"]
    maverick = configuration.get("is_maverick_scenario", False)
    args = (configuration["strategy"], configuration["dataset"], partition,
            parameter, configuration["run_number"], maverick)
    route = Path(get_times_from_route(*args))
    # exclude_maverick is not currently represented by the timing path.
    # Deduplicate the same physical path, without inventing a second run.
    if route in seen_routes:
        continue
    seen_routes.add(route)
    record = dict(zip(EXPERIMENT_COLUMNS, args[:5]))
    record["Timing_directory"] = str(route)
    expected_records.append(record)
    try:
        if not (route / "server").is_file():
            raise FileNotFoundError(f"Missing server timing file: {route / 'server'}")
        frames.append(get_times_dataframe(*args))
    except (OSError, ValueError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
        loading_records.append({**record, "Loading_problem": str(error)})
expected_experiments = pd.DataFrame(expected_records)
loading_problems = pd.DataFrame(loading_records, columns=[*EXPERIMENT_COLUMNS,
                                                        "Timing_directory", "Loading_problem"])
results_dataframe = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
    columns=[*EXPERIMENT_COLUMNS, "Round", "Evaluator"])
print(f"Expected timing directories: {len(expected_experiments)}; "
      f"loaded: {len(frames)}; loading failures: {len(loading_problems)}")


Expected timing directories: 300; loaded: 300; loading failures: 0


In [8]:
display(results_dataframe)

,BeforeLocalTraining,AfterLocalTraining,BeforeLocalPartialEvaluations,AfterLocalPartialEvaluations,BeforeEvaluation,AfterEvaluation,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Random_Seed,Round,Evaluator,BeforeSV,AfterSV,BeforeGradientRewards,AfterGradientRewards
0,1.641300,1.662031,2.767842,2.977860,3.005544,3.010548,FedAvg,har,dirichlet,0.1,1,1,Client_4,NaN,NaN,NaN,NaN
1,3.074733,3.096040,4.226964,4.439641,4.465643,4.469448,FedAvg,har,dirichlet,0.1,1,2,Client_4,NaN,NaN,NaN,NaN
2,4.530571,4.552372,5.622247,5.840920,5.869146,5.873316,FedAvg,har,dirichlet,0.1,1,3,Client_4,NaN,NaN,NaN,NaN
3,5.929469,5.949538,7.086211,7.419457,7.445408,7.449235,FedAvg,har,dirichlet,0.1,1,4,Client_4,NaN,NaN,NaN,NaN
4,7.506145,7.527246,8.887803,9.112928,9.139096,9.143585,FedAvg,har,dirichlet,0.1,1,5,Client_4,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68267,NaN,NaN,NaN,NaN,695.768811,695.888133,Scaffold,wids_dataset,manual,wids_energy_maverick_facility_type_uncategoriz...,5,46,Server,686.443433,695.454209,695.454517,695.763766
68268,NaN,NaN,NaN,NaN,711.080103,711.200859,Scaffold,wids_dataset,manual,wids_energy_maverick_facility_type_uncategoriz...,5,47,Server,701.836329,710.600856,710.601157,711.074387
68269,NaN,NaN,NaN,NaN,726.337901,726.458031,Scaffold,wids_dataset,manual,wids_energy_maverick_facility_type_uncategoriz...,5,48,Server,716.981661,726.026791,726.027059,726.332803
68270,NaN,NaN,NaN,NaN,741.824825,741.945376,Scaffold,wids_dataset,manual,wids_energy_maverick_facility_type_uncategoriz...,5,49,Server,732.422428,741.511078,741.511351,741.819794


In [9]:
results_dataframe.describe()

,BeforeLocalTraining,AfterLocalTraining,BeforeLocalPartialEvaluations,AfterLocalPartialEvaluations,BeforeEvaluation,AfterEvaluation,Random_Seed,Round,BeforeSV,AfterSV,BeforeGradientRewards,AfterGradientRewards
count,5.800100e+04,5.800100e+04,5.799700e+04,5.799700e+04,6.826800e+04,6.796800e+04,68272.000000,68272.000000,9971.000000,9971.000000,9971.000000,9971.000000
mean,-5.476412e+03,-5.476092e+03,-5.335570e+03,-5.333211e+03,-4.483470e+03,-4.503162e+03,3.030715,20.548688,312.747799,323.065856,323.066125,323.406431
std,1.043157e+05,1.043157e+05,1.031310e+05,1.031310e+05,9.507834e+04,9.528748e+04,1.412289,13.331584,423.072240,429.978212,429.978222,430.280269
min,-1.896556e+06,-1.896556e+06,-1.896545e+06,-1.896543e+06,-1.896543e+06,-1.896543e+06,1.000000,0.000000,0.823732,2.139655,2.140234,2.149927
25%,3.147212e+01,3.153977e+01,3.392662e+01,3.500135e+01,3.463817e+01,3.509833e+01,2.000000,9.000000,33.065262,35.329807,35.330015,35.414943
50%,9.653704e+01,9.687008e+01,1.074308e+02,1.097896e+02,1.114017e+02,1.134861e+02,3.000000,19.000000,121.909411,132.806056,132.806274,132.909409
75%,3.518547e+02,3.522901e+02,3.666699e+02,3.703239e+02,3.749017e+02,3.765105e+02,4.000000,31.000000,405.794118,419.618577,419.618833,420.169388
max,2.531014e+03,2.537045e+03,2.565025e+03,2.593681e+03,2.593688e+03,2.594603e+03,5.000000,50.000000,2470.883384,2497.479842,2497.480101,2500.160141


## Reusable timestamp audit

Run the loading and audit cells again after replacing experiment logs. The audit below preserves the original loaded dataframe and reports negative timestamps, large server/client offsets, missing client logs, negative operation durations, and unexpected round-count gaps. Its negative-only filter is followed by a stricter completeness filter used for the analysis.

Large offsets use a configurable threshold of max(5 seconds, twice the median server round spacing). Timestamp comparisons use shared rounds and tolerate up to five trailing client rounds. This is a diagnostic rule, not proof of a bug. The loader now reads each physical timing directory once: with/without-Maverick configurations currently resolve to the same directory and cannot be treated as separate measurements.


In [10]:
# Audit timestamps and exclude complete experiments with negative relative times.
# Keep results_dataframe unchanged so exclusions remain inspectable.
EXPERIMENT_COLUMNS = [
    "Aggregation_Process", "Dataset", "Type_of_partition",
    "Dirichlet/NamePartition", "Random_Seed",
]
TIME_COLUMNS = [column for column in results_dataframe.columns
                if column.startswith(("Before", "After"))]
NEGATIVE_TOLERANCE_SECONDS = 1e-6
# Diagnostic threshold, not an automatic exclusion rule:
# allow ordinary round-boundary delays, but flag large clock/run mismatches.
MIN_OFFSET_TOLERANCE_SECONDS = 5.0
OFFSET_TOLERANCE_ROUNDS = 2.0
ALLOWED_EXTRA_CLIENT_ROUNDS = 5


def audit_training_times(frame):
    records = []
    for key, experiment in frame.groupby(EXPERIMENT_COLUMNS, dropna=False, sort=False):
        # The loader currently repeats some Maverick runs. Do not let exact
        # duplicate rows weight the diagnostic medians more than once.
        experiment = experiment.drop_duplicates()
        server = experiment.loc[experiment["Evaluator"].eq("Server")]
        clients = experiment.loc[~experiment["Evaluator"].eq("Server")]
        record = dict(zip(EXPERIMENT_COLUMNS, key))
        values = experiment[TIME_COLUMNS]
        record["Has_negative_timestamp"] = bool(
            values.lt(-NEGATIVE_TOLERANCE_SECONDS).any().any())
        record["Minimum_timestamp_s"] = values.min().min()
        record["Negative_evaluators"] = ", ".join(sorted(
            experiment.loc[values.lt(-NEGATIVE_TOLERANCE_SECONDS).any(axis=1),
                           "Evaluator"].unique()))
        record["Missing_server"] = server.empty
        record["Missing_clients"] = clients.empty
        record["Large_time_offset"] = False
        record["Unexpected_round_gap"] = False
        record["Negative_duration"] = False
        for before in TIME_COLUMNS:
            after = "After" + before[len("Before"):]
            if before.startswith("Before") and after in TIME_COLUMNS:
                record["Negative_duration"] |= bool(
                    (experiment[after] - experiment[before])
                    .lt(-NEGATIVE_TOLERANCE_SECONDS).any())
        if not server.empty and not clients.empty:
            server_start = server[TIME_COLUMNS].min().min()
            server_end = server[TIME_COLUMNS].max().max()
            round_steps = (server.groupby("Round")["BeforeEvaluation"].min()
                           .sort_index().diff())
            typical_round_s = round_steps.loc[round_steps.gt(0)].median()
            tolerance = max(MIN_OFFSET_TOLERANCE_SECONDS,
                            OFFSET_TOLERANCE_ROUNDS * typical_round_s
                            if pd.notna(typical_round_s) else 0.0)
            # Compare only shared rounds. Trailing client-only rounds (up to
            # five) are ignored for the server/client timestamp comparison.
            shared = clients.loc[clients["Round"].isin(server["Round"])]
            client_start = shared[TIME_COLUMNS].min().min()
            client_end = shared[TIME_COLUMNS].max().max()
            before_start = max(0.0, server_start - client_start)
            after_end = max(0.0, client_end - server_end)
            round_gaps = clients.groupby("Evaluator")["Round"].max() - server["Round"].max()
            record.update({
                "Server_elapsed_s": server_end - server_start,
                "Typical_round_s": typical_round_s,
                "Offset_tolerance_s": tolerance,
                "Client_before_server_window_s": before_start,
                "Client_after_server_window_s": after_end,
                "Min_extra_client_rounds": round_gaps.min(),
                "Max_extra_client_rounds": round_gaps.max(),
                "Large_time_offset": before_start > tolerance or after_end > tolerance,
                "Unexpected_round_gap": bool(
                    (round_gaps.lt(0) | round_gaps.gt(ALLOWED_EXTRA_CLIENT_ROUNDS)).any()),
            })
        records.append(record)
    return pd.DataFrame(records)


time_audit = audit_training_times(results_dataframe)
excluded_experiments = time_audit.loc[time_audit["Has_negative_timestamp"]].copy()
excluded_keys = pd.MultiIndex.from_frame(excluded_experiments[EXPERIMENT_COLUMNS])
row_keys = pd.MultiIndex.from_frame(results_dataframe[EXPERIMENT_COLUMNS])
analysis_times_dataframe = results_dataframe.loc[~row_keys.isin(excluded_keys)].copy()
# Other diagnostics are displayed for review; they do not remove more runs.
other_suspicious_experiments = time_audit.loc[
    ~time_audit["Has_negative_timestamp"] & time_audit[
        ["Large_time_offset", "Unexpected_round_gap", "Negative_duration",
         "Missing_server", "Missing_clients"]].any(axis=1)
].copy()
print(f"Distinct experiments: {len(time_audit)}")
print(f"Excluded for negative timestamps: {len(excluded_experiments)}")
print(f"Remaining experiments: {len(time_audit) - len(excluded_experiments)}")
print(f"Other suspicious experiments: {len(other_suspicious_experiments)}")
display(excluded_experiments)
display(other_suspicious_experiments)


Distinct experiments: 300
Excluded for negative timestamps: 1
Remaining experiments: 299
Other suspicious experiments: 3


,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Random_Seed,Has_negative_timestamp,Minimum_timestamp_s,Negative_evaluators,Missing_server,Missing_clients,Large_time_offset,Unexpected_round_gap,Negative_duration,Server_elapsed_s,Typical_round_s,Offset_tolerance_s,Client_before_server_window_s,Client_after_server_window_s,Min_extra_client_rounds,Max_extra_client_rounds
166,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,5,True,-1.896556e+06,"Client_0, Client_1, Client_2, Client_3",False,False,True,True,False,1553.255016,30.771361,61.542721,1.896556e+06,0.0,-6.0,-6.0


,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Random_Seed,Has_negative_timestamp,Minimum_timestamp_s,Negative_evaluators,Missing_server,Missing_clients,Large_time_offset,Unexpected_round_gap,Negative_duration,Server_elapsed_s,Typical_round_s,Offset_tolerance_s,Client_before_server_window_s,Client_after_server_window_s,Min_extra_client_rounds,Max_extra_client_rounds
138,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,1,False,0.0,,False,True,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,2,False,0.0,,False,True,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,4,False,0.0,,False,True,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The counts below are recomputed from the current files. Rerun the notebook after rerunning experiments; do not rely on counts from an earlier execution.


## Complete experiments and rerun summary

The primary analysis requires server and all expected client logs, complete training/SV timing pairs in every server training round, and valid timestamp diagnostics. An entire experiment is excluded if any required round is incomplete. Initialization (round 0) and up to five client-only trailing rounds are not analysed. Expected clients are taken from ordinary `client_<id>_y_training.csv` files, excluding Maverick-excluded variants. If partition metadata is absent, completeness cannot be established and the run is held out.

`missing_experiments` is the detailed rerun list. `missing_experiments_summary` counts exclusion reasons; one experiment may have several reasons. `analysis_coverage` shows expected, retained, and excluded seeds for each configuration. `analysis_times_dataframe` contains only retained experiments and matched training rounds.


In [11]:
import re
import numpy as np


def partition_data_directory(dataset, partition, parameter, seed):
    base = Path(CONST_ROUTE_MAIN_DIR) / "data" / "partitioned_training_data" / partition
    if partition == "manual":
        return base / str(parameter) / str(seed)
    return base / ("dataset_" + dataset) / ("alpha_" + str(parameter)) / str(seed)


def partition_client_ids(directory):
    return sorted(match.group(1) for file in directory.glob("client_*_y_training.csv")
                  if (match := re.fullmatch(r"client_(\d+)_y_training\.csv", file.name)))


SERVER_REQUIRED = ["BeforeEvaluation", "AfterEvaluation", "BeforeSV", "AfterSV"]
CLIENT_REQUIRED = ["BeforeLocalTraining", "AfterLocalTraining",
                   "BeforeLocalPartialEvaluations", "AfterLocalPartialEvaluations"]
quality_records = []
round_issue_records = []
for _, expected in expected_experiments.iterrows():
    key = tuple(expected[column] for column in EXPERIMENT_COLUMNS)
    mask = pd.Series(True, index=results_dataframe.index)
    for column, value in zip(EXPERIMENT_COLUMNS, key):
        mask &= results_dataframe[column].eq(value)
    experiment = results_dataframe.loc[mask].drop_duplicates()
    reasons = []
    record = expected.to_dict()
    directory = partition_data_directory(key[1], key[2], key[3], key[4])
    expected_clients = partition_client_ids(directory)
    record["Expected_clients"] = len(expected_clients)
    record["Data_directory"] = str(directory)
    if not expected_clients:
        reasons.append("Missing partition metadata")
    if experiment.empty:
        reasons.append("Missing or unreadable timing files")
        record.update({"Observed_clients": 0, "Missing_client_ids": ", ".join(expected_clients),
                       "Server_training_rounds": 0})
    else:
        audit_row = time_audit.set_index(EXPERIMENT_COLUMNS).loc[key]
        for flag, label in {
            "Has_negative_timestamp": "Negative timestamps",
            "Large_time_offset": "Large server/client offset",
            "Unexpected_round_gap": "Unexpected round gap",
            "Negative_duration": "Negative operation duration",
            "Missing_server": "Missing server log",
            "Missing_clients": "Missing client logs",
        }.items():
            if audit_row[flag]:
                reasons.append(label)
        server = experiment.loc[experiment.Evaluator.eq("Server")].sort_values("Round")
        clients = experiment.loc[~experiment.Evaluator.eq("Server")]
        observed = set(clients.Evaluator.str.removeprefix("Client_"))
        missing_ids = sorted(set(expected_clients) - observed)
        record.update({"Observed_clients": len(observed),
                       "Missing_client_ids": ", ".join(missing_ids),
                       "Server_training_rounds": int(server.Round.gt(0).sum())})
        if missing_ids and "Missing client logs" not in reasons:
            reasons.append("Missing client logs")
        if observed - set(expected_clients) and expected_clients:
            reasons.append("Unexpected client logs")
        if experiment.duplicated(["Evaluator", "Round"]).any():
            reasons.append("Conflicting duplicate round logs")
        if not server.Round.eq(0).any():
            reasons.append("Missing server initialization")
        training_rounds = server.loc[server.Round.gt(0), "Round"].tolist()
        if not training_rounds:
            reasons.append("No server training rounds")
        elif training_rounds != list(range(1, int(max(training_rounds)) + 1)):
            reasons.append("Missing server rounds")
        for _, row in server.loc[server.Round.gt(0)].iterrows():
            rnd = row.Round
            round_clients = clients.loc[clients.Round.eq(rnd)]
            missing_in_round = set(expected_clients) - set(
                round_clients.Evaluator.str.removeprefix("Client_"))
            server_complete = row.reindex(SERVER_REQUIRED).notna().all()
            client_complete = (not round_clients.empty and
                               round_clients.reindex(columns=CLIENT_REQUIRED).notna().all().all())
            if missing_in_round or not server_complete or not client_complete:
                round_issue_records.append({**dict(zip(EXPERIMENT_COLUMNS, key)),
                    "Round": rnd, "Missing_client_ids": ", ".join(sorted(missing_in_round)),
                    "Missing_server_pair": not server_complete,
                    "Missing_client_pair": not client_complete})
                if "Incomplete round timings" not in reasons:
                    reasons.append("Incomplete round timings")
    record["Problems"] = "; ".join(dict.fromkeys(reasons))
    record["Eligible"] = not reasons
    quality_records.append(record)

experiment_quality = pd.DataFrame(quality_records)
missing_experiments = experiment_quality.loc[~experiment_quality.Eligible].copy()
missing_round_timings = pd.DataFrame(round_issue_records)
missing_experiments_summary = (
    missing_experiments.assign(Reason=missing_experiments.Problems.str.split("; "))
    .explode("Reason").groupby(["Aggregation_Process", "Dataset", "Reason"])
    .size().rename("Experiments").reset_index())
CONFIG_COLUMNS = EXPERIMENT_COLUMNS[:-1]
analysis_coverage = experiment_quality.groupby(CONFIG_COLUMNS, dropna=False).agg(
    Expected_experiments=("Eligible", "size"), Retained_experiments=("Eligible", "sum"),
    Expected_seeds=("Random_Seed", lambda x: ", ".join(map(str, sorted(x)))),
).reset_index()
excluded_seed_summary = missing_experiments.groupby(CONFIG_COLUMNS, dropna=False)["Random_Seed"].agg(
    lambda x: ", ".join(map(str, sorted(x)))).rename("Excluded_seeds").reset_index()
analysis_coverage = analysis_coverage.merge(excluded_seed_summary, on=CONFIG_COLUMNS, how="left")
analysis_coverage["Excluded_seeds"] = analysis_coverage.Excluded_seeds.fillna("")
analysis_coverage["Excluded_experiments"] = (analysis_coverage.Expected_experiments -
                                              analysis_coverage.Retained_experiments)
eligible_keys = pd.MultiIndex.from_frame(experiment_quality.loc[
    experiment_quality.Eligible, EXPERIMENT_COLUMNS])
row_keys = pd.MultiIndex.from_frame(results_dataframe[EXPERIMENT_COLUMNS])
analysis_times_dataframe = results_dataframe.loc[row_keys.isin(eligible_keys)].drop_duplicates().copy()
server_round_keys = pd.MultiIndex.from_frame(analysis_times_dataframe.loc[
    analysis_times_dataframe.Evaluator.eq("Server") & analysis_times_dataframe.Round.gt(0),
    [*EXPERIMENT_COLUMNS, "Round"]])
# Keep initialization separately for the first round's elapsed-time baseline.
analysis_initialization = analysis_times_dataframe.loc[
    analysis_times_dataframe.Evaluator.eq("Server") & analysis_times_dataframe.Round.eq(0)].copy()
analysis_times_dataframe = analysis_times_dataframe.loc[pd.MultiIndex.from_frame(
    analysis_times_dataframe[[*EXPERIMENT_COLUMNS, "Round"]]).isin(server_round_keys)].copy()
print(f"Expected: {len(experiment_quality)}; retained: {int(experiment_quality.Eligible.sum())}; "
      f"excluded: {len(missing_experiments)}")
display(missing_experiments_summary)
display(missing_experiments[[*EXPERIMENT_COLUMNS, "Problems", "Missing_client_ids",
                             "Timing_directory"]])
display(analysis_coverage)
if not loading_problems.empty:
    display(loading_problems)


Expected: 300; retained: 296; excluded: 4


,Aggregation_Process,Dataset,Reason,Experiments
0,FedProx,edge-iot-coreset,Incomplete round timings,4
1,FedProx,edge-iot-coreset,Large server/client offset,1
2,FedProx,edge-iot-coreset,Missing client logs,4
3,FedProx,edge-iot-coreset,Negative timestamps,1
4,FedProx,edge-iot-coreset,Unexpected round gap,1


,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Random_Seed,Problems,Missing_client_ids,Timing_directory
138,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,1,Missing client logs; Incomplete round timings,"0, 1, 2, 3, 4",../../results/times/FedProx/edge-iot-coreset/m...
145,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,2,Missing client logs; Incomplete round timings,"0, 1, 2, 3, 4",../../results/times/FedProx/edge-iot-coreset/m...
159,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,4,Missing client logs; Incomplete round timings,"0, 1, 2, 3, 4",../../results/times/FedProx/edge-iot-coreset/m...
166,FedProx,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,5,Negative timestamps; Large server/client offse...,4,../../results/times/FedProx/edge-iot-coreset/m...


,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Expected_experiments,Retained_experiments,Expected_seeds,Excluded_seeds,Excluded_experiments
0,FedAvg,edge-iot-coreset,dirichlet,0.1,5,5,"1, 2, 3, 4, 5",,0
1,FedAvg,edge-iot-coreset,dirichlet,10,5,5,"1, 2, 3, 4, 5",,0
2,FedAvg,edge-iot-coreset,dirichlet,1000,5,5,"1, 2, 3, 4, 5",,0
3,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,5,5,"1, 2, 3, 4, 5",,0
4,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Only_Normal,5,5,"1, 2, 3, 4, 5",,0
5,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,5,5,"1, 2, 3, 4, 5",,0
6,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_sql_injection,5,5,"1, 2, 3, 4, 5",,0
7,FedAvg,har,dirichlet,0.1,5,5,"1, 2, 3, 4, 5",,0
8,FedAvg,har,dirichlet,10,5,5,"1, 2, 3, 4, 5",,0
9,FedAvg,har,dirichlet,1000,5,5,"1, 2, 3, 4, 5",,0


## Round durations and SV overhead

For each retained round, server SV is `AfterSV - BeforeSV`, and client SV is `max(AfterLocalPartialEvaluations) - min(BeforeLocalPartialEvaluations)` across clients. Local training uses the analogous interval spanning all clients, not the sum of their parallel work.

A full round ends at the latest recorded server or client completion timestamp for that round. It starts at the preceding full round's completion; the first round starts at the server's initialization `BeforeEvaluation`. Client partial evaluations occur after the same-numbered server evaluation in these logs, so a server-only boundary would omit client SV work. Combined SV overhead is the union of the server and client SV intervals (so overlap is counted once). Estimated time without SV is elapsed round time minus that union. This subtraction does not measure a retraining run with SV disabled. Both SV/round percentages and SV/local-training percentages are reported; the latter can exceed 100%.

Summaries first aggregate rounds within each experiment, then summarize experiments so longer runs do not receive more weight. Configuration tables keep partitions separate; the compact plots summarize the retained mix of partitions.

`experiment_times.Rounds` counts analysed training rounds per experiment. `configuration_round_summary` reports experiment count, total rounds, and the minimum/mean/maximum rounds per experiment. Initialization and extra client-only rounds are excluded.


In [12]:
round_records = []
for key, experiment in analysis_times_dataframe.groupby(EXPERIMENT_COLUMNS, sort=False, dropna=False):
    server = experiment.loc[experiment.Evaluator.eq("Server")].sort_values("Round")
    init = analysis_initialization
    for column, value in zip(EXPERIMENT_COLUMNS, key):
        init = init.loc[init[column].eq(value)]
    previous_end = init.BeforeEvaluation.iloc[0]
    for _, row in server.iterrows():
        clients = experiment.loc[~experiment.Evaluator.eq("Server") & experiment.Round.eq(row.Round)]
        completion_columns = [column for column in TIME_COLUMNS if column.startswith("After")]
        latest_client_end = clients[completion_columns].max().max()
        latest_server_end = row.reindex(completion_columns).max()
        start, end = previous_end, max(latest_server_end, latest_client_end)
        previous_end = end
        s0, s1 = row.BeforeSV, row.AfterSV
        c0, c1 = clients.BeforeLocalPartialEvaluations.min(), clients.AfterLocalPartialEvaluations.max()
        t0, t1 = clients.BeforeLocalTraining.min(), clients.AfterLocalTraining.max()
        overlap = max(0.0, min(s1, c1) - max(s0, c0))
        server_sv, client_sv = s1 - s0, c1 - c0
        combined_sv = server_sv + client_sv - overlap
        elapsed, training = end - start, t1 - t0
        valid = (elapsed > 0 and training > 0 and min(s0, c0, t0) >= start - 1e-6
                 and max(s1, c1, t1) <= end + 1e-6 and combined_sv <= elapsed + 1e-6)
        round_records.append({**dict(zip(EXPERIMENT_COLUMNS, key)), "Round": row.Round,
            "Clients": clients.Evaluator.nunique(), "Round_elapsed_s": elapsed,
            "Local_training_s": training, "Server_SV_s": server_sv,
            "Client_SV_s": client_sv, "SV_overlap_s": overlap,
            "Combined_SV_s": combined_sv, "Estimated_without_SV_s": elapsed - combined_sv,
            "Valid_round_window": valid})
round_times = pd.DataFrame(round_records, columns=[*EXPERIMENT_COLUMNS, "Round", "Clients",
    "Round_elapsed_s", "Local_training_s", "Server_SV_s", "Client_SV_s", "SV_overlap_s",
    "Combined_SV_s", "Estimated_without_SV_s", "Valid_round_window"])
invalid_round_windows = round_times.loc[~round_times.Valid_round_window.astype(bool)].copy()
# Hold out whole runs if their stages do not fit the measured round boundaries.
invalid_keys = pd.MultiIndex.from_frame(invalid_round_windows[EXPERIMENT_COLUMNS].drop_duplicates())
round_times = round_times.loc[~pd.MultiIndex.from_frame(round_times[EXPERIMENT_COLUMNS])
                             .isin(invalid_keys)].copy()
for stage in ["Server_SV", "Client_SV", "Combined_SV"]:
    round_times[stage + "_pct_round"] = 100 * round_times[stage + "_s"] / round_times.Round_elapsed_s
    round_times[stage + "_pct_training"] = 100 * round_times[stage + "_s"] / round_times.Local_training_s
DURATION_COLUMNS = ["Round_elapsed_s", "Local_training_s", "Server_SV_s", "Client_SV_s",
                    "Combined_SV_s", "Estimated_without_SV_s"]
experiment_times = round_times.groupby(EXPERIMENT_COLUMNS, dropna=False).agg(
    Rounds=("Round", "size"), Clients=("Clients", "first"),
    **{column: (column, "sum") for column in DURATION_COLUMNS}).reset_index()
for column in DURATION_COLUMNS:
    experiment_times["Mean_" + column] = experiment_times[column] / experiment_times.Rounds
for stage in ["Server_SV", "Client_SV", "Combined_SV"]:
    experiment_times[stage + "_pct_round"] = (100 * experiment_times[stage + "_s"] /
                                              experiment_times.Round_elapsed_s)
    experiment_times[stage + "_pct_training"] = (100 * experiment_times[stage + "_s"] /
                                                 experiment_times.Local_training_s)
SUMMARY_METRICS = ["Mean_" + column for column in DURATION_COLUMNS] + [
    stage + suffix for stage in ["Server_SV", "Client_SV", "Combined_SV"]
    for suffix in ["_pct_round", "_pct_training"]]
configuration_round_summary = experiment_times.groupby(CONFIG_COLUMNS, dropna=False).agg(
    Experiments=("Rounds", "size"), Total_rounds=("Rounds", "sum"),
    Rounds_per_experiment_min=("Rounds", "min"),
    Rounds_per_experiment_mean=("Rounds", "mean"),
    Rounds_per_experiment_max=("Rounds", "max")).reset_index()
configuration_timing_summary = experiment_times.groupby(CONFIG_COLUMNS, dropna=False)[
    SUMMARY_METRICS].agg(["mean", "std", "count"])
# Propagate additional boundary exclusions into the rerun and coverage tables.
if not invalid_round_windows.empty:
    invalid_details = invalid_round_windows[EXPERIMENT_COLUMNS].drop_duplicates()
    invalid_details["Problems"] = "Stages outside measured round window"
    missing_experiments = pd.concat([missing_experiments, invalid_details], ignore_index=True)
    missing_experiments_summary = (missing_experiments.assign(
        Reason=missing_experiments.Problems.str.split("; ")).explode("Reason")
        .groupby(["Aggregation_Process", "Dataset", "Reason"]).size()
        .rename("Experiments").reset_index())
    analysis_times_dataframe = analysis_times_dataframe.loc[~pd.MultiIndex.from_frame(
        analysis_times_dataframe[EXPERIMENT_COLUMNS]).isin(invalid_keys)].copy()
    retained = experiment_times.groupby(CONFIG_COLUMNS, dropna=False).size().rename("Final_retained")
    analysis_coverage = analysis_coverage.merge(retained.reset_index(), on=CONFIG_COLUMNS, how="left")
    analysis_coverage["Retained_experiments"] = analysis_coverage.Final_retained.fillna(0).astype(int)
    analysis_coverage["Excluded_experiments"] = (analysis_coverage.Expected_experiments -
                                                analysis_coverage.Retained_experiments)
    excluded = missing_experiments.groupby(CONFIG_COLUMNS, dropna=False).Random_Seed.agg(
        lambda x: ", ".join(map(str, sorted(set(x)))))
    analysis_coverage = analysis_coverage.drop(columns=["Excluded_seeds", "Final_retained"]).merge(
        excluded.rename("Excluded_seeds").reset_index(), on=CONFIG_COLUMNS, how="left")
    analysis_coverage["Excluded_seeds"] = analysis_coverage.Excluded_seeds.fillna("")
    display(invalid_round_windows)
    display(missing_experiments_summary)
    display(analysis_coverage)
print(f"Analysed experiments: {len(experiment_times)}; rounds: {len(round_times)}")
display(configuration_round_summary)
display(configuration_timing_summary)
display(experiment_times)


Analysed experiments: 296; rounds: 9771


,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Experiments,Total_rounds,Rounds_per_experiment_min,Rounds_per_experiment_mean,Rounds_per_experiment_max
0,FedAvg,edge-iot-coreset,dirichlet,0.1,5,56,8,11.2,16
1,FedAvg,edge-iot-coreset,dirichlet,10,5,137,10,27.4,50
2,FedAvg,edge-iot-coreset,dirichlet,1000,5,250,50,50.0,50
3,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,5,250,50,50.0,50
4,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Only_Normal,5,134,11,26.8,39
5,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,5,242,42,48.4,50
6,FedAvg,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_sql_injection,5,68,2,13.6,39
7,FedAvg,har,dirichlet,0.1,5,220,36,44.0,50
8,FedAvg,har,dirichlet,10,5,129,20,25.8,31
9,FedAvg,har,dirichlet,1000,5,86,10,17.2,29


Mean_Round_elapsed_s  \
                                                                                                                          mean   
Aggregation_Process Dataset          Type_of_partition Dirichlet/NamePartition                                                   
FedAvg              edge-iot-coreset dirichlet         0.1                                                           33.578859   
                                                       10                                                            35.377201   
                                                       1000                                                          34.595889   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                        32.682115   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                        48.001949   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                           36.811983   
                                                       edgeiot_coreset_1_Maverick_sql_injection                      36.487360   
                    har              dirichlet         0.1                                                            1.549774   
                                                       10                                                             1.688763   
                                                       1000                                                           1.801425   
                                     manual            HAR_1_Maverick_Laying                                          1.771446   
                                                       HAR_1_Maverick_Laying_Balanced                                 1.583803   
                                                       HAR_1_Maverick_WalkingUpstairs                                 1.960542   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced                        1.486741   
                    wids_dataset     manual            wids_energy_feature_skew_building_type                         7.930030   
                                                       wids_energy_feature_skew_state_factor                         11.676192   
                                                       wids_energy_iid_sampling                                      10.336855   
                                                       wids_energy_maverick_facility_type_grocery_store              14.081483   
                                                       wids_energy_maverick_facility_type_uncategorize...            13.202644   
                                                       wids_energy_non_iid_by_label                                   8.531475   
FedProx             edge-iot-coreset dirichlet         0.1                                                           32.676999   
                                                       10                                                            34.006684   
                                                       1000                                                          34.195715   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                        33.462053   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                        54.414796   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                           41.637146   
                                                       edgeiot_coreset_1_Maverick_sql_injection                      34.693042   
                    har              dirichlet         0.1                                                            1.639148   
                                                       10                   

,Aggregation_Process,Dataset,Type_of_partition,Dirichlet/NamePartition,Random_Seed,Rounds,Clients,Round_elapsed_s,Local_training_s,Server_SV_s,...,Mean_Server_SV_s,Mean_Client_SV_s,Mean_Combined_SV_s,Mean_Estimated_without_SV_s,Server_SV_pct_round,Server_SV_pct_training,Client_SV_pct_round,Client_SV_pct_training,Combined_SV_pct_round,Combined_SV_pct_training
0,FedAvg,edge-iot-coreset,dirichlet,0.1,1,12,5,372.025426,6.887255,293.447222,...,24.453935,4.470170,28.924105,2.078014,78.878270,4260.728162,14.418918,778.859519,93.297189,5039.587680
1,FedAvg,edge-iot-coreset,dirichlet,0.1,2,8,5,297.380264,5.652700,225.838961,...,28.229870,5.282921,33.512791,3.659742,75.942821,3995.240220,14.211893,747.666803,90.154714,4742.907024
2,FedAvg,edge-iot-coreset,dirichlet,0.1,3,16,5,525.219573,20.336037,351.884748,...,21.992797,7.285609,29.278405,3.547818,66.997646,1730.350673,22.194477,573.217578,89.192122,2303.568251
3,FedAvg,edge-iot-coreset,dirichlet,0.1,4,12,5,445.590965,10.949586,325.185660,...,27.098805,6.157746,33.256551,3.876030,72.978513,2969.844428,16.583134,674.846962,89.561647,3644.691390
4,FedAvg,edge-iot-coreset,dirichlet,0.1,5,8,5,238.086725,5.243947,184.708565,...,23.088571,4.475161,27.563732,2.197109,77.580371,3522.319423,15.037079,682.716447,92.617450,4205.035870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,Scaffold,wids_dataset,manual,wids_energy_non_iid_by_label,1,50,6,433.412119,9.131657,315.227872,...,6.304557,1.779696,8.084254,0.583989,72.731670,3452.033690,20.531224,974.465157,93.262894,4426.498847
292,Scaffold,wids_dataset,manual,wids_energy_non_iid_by_label,2,27,6,178.418276,3.125921,140.069180,...,5.187747,1.047968,6.235716,0.372368,78.506072,4480.892452,15.858884,905.177825,94.364956,5386.070277
293,Scaffold,wids_dataset,manual,wids_energy_non_iid_by_label,3,37,6,226.985315,4.397718,175.036977,...,4.730729,1.059193,5.789922,0.344816,77.113789,3980.177411,17.265501,891.147435,94.379289,4871.324847
294,Scaffold,wids_dataset,manual,wids_energy_non_iid_by_label,4,36,6,253.198735,5.958518,181.939584,...,5.053877,1.538861,6.592738,0.440560,71.856435,3053.437075,21.879645,929.744400,93.736080,3983.181474


## Compact timing plots

Two panels show mean elapsed seconds per round (total SV as the leftmost bar, measured with SV, and estimated without SV), and SV's percentage of elapsed round time. Each point in the underlying summaries is an experiment; error bars in the percentage panel show variation across retained experiments, including partition differences.

Figures use Plotly, are displayed interactively in the notebook, and are exported as named vector PDFs to the `images` folder beside this notebook. Plotly and Kaleido are required for PDF export.

Each plot is exported and displayed separately. The seven PDFs are: elapsed round time, SV share of round time, total SV versus clients, total SV versus rounds, total SV versus test-set size, total SV summary by dataset, and total SV summary by test-set size. Error bars remain; their meaning is documented in the analysis rather than appended to axis labels.


In [16]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from plotly.colors import qualitative
import tempfile

pio.renderers.default = "plotly_mimetype"
IMAGE_DIRECTORY = Path(CONST_ROUTE_MAIN_DIR) / "notebooks" / "NotebooksActualExperimentation" / "images"
IMAGE_DIRECTORY.mkdir(parents=True, exist_ok=True)


def save_analysis_figure(figure, filename, width, height):
    # Keep exporter temporary files inside the authorised images directory.
    previous_tempdir = tempfile.tempdir
    tempfile.tempdir = str(IMAGE_DIRECTORY.resolve())
    previous_browser = os.environ.get("BROWSER_PATH")
    if previous_browser is None and Path("/opt/google/chrome/chrome").is_file():
        # Use the Chrome binary directly rather than its desktop launcher.
        os.environ["BROWSER_PATH"] = "/opt/google/chrome/chrome"
    try:
        output = (IMAGE_DIRECTORY / filename).with_suffix(".pdf")
        # Plotly's export API supports both legacy and current Kaleido versions;
        # do not call Kaleido's version-specific write_fig_sync API directly.
        figure.write_image(str(output), format="pdf", width=width, height=height)
    finally:
        tempfile.tempdir = previous_tempdir
        if previous_browser is None:
            os.environ.pop("BROWSER_PATH", None)
        else:
            os.environ["BROWSER_PATH"] = previous_browser
    print(f"Saved {output}")
    figure.show()


def save_single_panel(source_figure, column, filename):
    """Export one subplot as an independent Plotly figure with its own axes."""
    axis_suffix = "" if column == 1 else str(column)
    x_reference = "x" + axis_suffix
    figure = go.Figure()
    for source_trace in source_figure.data:
        if (source_trace.xaxis or "x") == x_reference:
            trace = source_trace.to_plotly_json()
            trace.pop("xaxis", None)
            trace.pop("yaxis", None)
            # Each independent figure needs its own legend.
            trace["showlegend"] = True
            figure.add_trace(trace)
    x_axis = source_figure.layout["xaxis" + axis_suffix].to_plotly_json()
    y_axis = source_figure.layout["yaxis" + axis_suffix].to_plotly_json()
    for axis, anchor in [(x_axis, "y"), (y_axis, "x")]:
        axis["domain"] = [0, 1]
        axis["anchor"] = anchor
        axis.pop("matches", None)
        if isinstance(axis.get("title"), dict):
            for phrase in [" (mean ± SD)", " (mean + std)", " (mean ± std)"]:
                axis["title"]["text"] = axis["title"].get("text", "").replace(phrase, "")
    angled_labels = x_axis.get("tickangle") == -45
    show_legend = not (len(figure.data) == 1 and figure.data[0].type == "bar")
    figure.update_layout(template="plotly_white", xaxis=x_axis, yaxis=y_axis, showlegend=show_legend,
        title=source_figure.layout.annotations[column - 1].text,
        barmode=source_figure.layout.barmode or "group",
        legend=dict(orientation="h", yanchor="top", y=-0.48 if angled_labels else -0.25, x=0),
        margin=dict(t=85, b=245 if angled_labels else 175, l=95, r=35))
    save_analysis_figure(figure, filename, 1000, 760 if angled_labels else 700)
    return figure


if experiment_times.empty:
    print("No complete experiments are available for plotting.")
else:
    plot_summary = experiment_times.groupby(["Dataset", "Aggregation_Process"])[
        ["Mean_Combined_SV_s", "Mean_Round_elapsed_s", "Mean_Estimated_without_SV_s",
         "Server_SV_pct_round", "Client_SV_pct_round", "Combined_SV_pct_round"]].agg(["mean", "std"])
    plot_round_summary = experiment_times.groupby(["Dataset", "Aggregation_Process"]).agg(
        Experiments=("Rounds", "size"), Total_rounds=("Rounds", "sum"),
        Rounds_min=("Rounds", "min"), Rounds_max=("Rounds", "max"))
    display(plot_round_summary)
    labels = [f"{dataset}<br>{strategy}" for dataset, strategy in plot_summary.index]
    timing_figure = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
        subplot_titles=["Elapsed round time", "SV share"])
    for metric, label, color in [
        ("Mean_Combined_SV_s", "Total SV (server + clients)", "#2ca02c"),
        ("Mean_Round_elapsed_s", "Measured with SV", "#1f77b4"),
        ("Mean_Estimated_without_SV_s", "Estimated without SV", "#ff7f0e")]:
        timing_figure.add_trace(go.Bar(x=labels, y=plot_summary[(metric, "mean")],
            name=label, marker_color=color), row=1, col=1)
    for metric, label, color in [("Server_SV_pct_round", "Server SV share", "#1f77b4"),
                                 ("Client_SV_pct_round", "Client SV share", "#ff7f0e"),
                                 ("Combined_SV_pct_round", "Combined SV share", "#2ca02c")]:
        timing_figure.add_trace(go.Scatter(x=labels, y=plot_summary[(metric, "mean")],
            mode="markers", name=label, marker=dict(color=color, size=9),
            error_y=dict(type="data", array=plot_summary[(metric, "std")].fillna(0))), row=1, col=2)
    timing_figure.update_yaxes(title_text="Mean elapsed seconds per round", row=1, col=1)
    timing_figure.update_yaxes(title_text="SV / elapsed round time (%)", row=1, col=2)
    timing_figure.update_xaxes(tickangle=-45, categoryorder="array", categoryarray=labels)
    timing_figure.update_layout(template="plotly_white", barmode="group",
        title=f"{len(experiment_times)} experiments; {int(experiment_times.Rounds.sum())} training rounds",
        legend=dict(orientation="h", yanchor="top", y=-0.50, x=0),
        margin=dict(t=90, b=220, l=70, r=40))
    elapsed_round_figure = save_single_panel(timing_figure, 1, "sv_elapsed_round_time.pdf")
    round_share_figure = save_single_panel(timing_figure, 2, "sv_round_time_share.pdf")


Experiments  Total_rounds  Rounds_min  \
Dataset          Aggregation_Process                                          
edge-iot-coreset FedAvg                        35          1137           2   
                 FedProx                       31           670           2   
                 Scaffold                      35          1090           2   
har              FedAvg                        35          1055          10   
                 FedProx                       35          1223          10   
                 Scaffold                      35          1200          16   
wids_dataset     FedAvg                        30          1197           4   
                 FedProx                       30          1054           5   
                 Scaffold                      30          1145           2   

                                      Rounds_max  
Dataset          Aggregation_Process              
edge-iot-coreset FedAvg                       50  
                 FedProx                      50  
                 Scaffold                     50  
har              FedAvg                       50  
                 FedProx                      50  
                 Scaffold                     50  
wids_dataset     FedAvg                       50  
                 FedProx                      50  
                 Scaffold                     50

Saved ../../notebooks/NotebooksActualExperimentation/images/sv_elapsed_round_time.pdf


Saved ../../notebooks/NotebooksActualExperimentation/images/sv_round_time_share.pdf


## Dataset size and exploratory scalability

Dataset sizes are counted from partitioned label CSVs, counting ordinary training/test files once per client and excluding Maverick-excluded variants. This measures the actual partition used by the included-Maverick timing path. Training sample count is relevant to model fitting; combined client test sample count is relevant to distributed model inference. The separately reported global test set is relevant to server evaluation. Metadata reads are cached per dataset/partition/seed within this execution.

All three panels use the same total combined server + client SV seconds summed over all analysed rounds for each experiment. The panels compare number of clients, analysed rounds per experiment, and total client test samples; their timing values are identical for the same experiment. The summary also reports total analysed rounds and the minimum, mean, and maximum rounds per experiment. These are exploratory comparisons: dataset identity, feature count, model, partition, client count, and hardware may differ together. Five versus six clients across different datasets does not establish an isolated client scalability effect. Repeated seeds provide runtime variation, not additional independent dataset sizes. No complexity exponent is fitted to these observations.

Each point is one experiment (one seed). Its total is the sum of its measured round intervals, not a fixed round count multiplied by a global mean. Runs with different numbers of analysed rounds may have different totals for that reason.

In the round-count and test-size figures, marker shape denotes dataset (HAR: circle, Edge-IoT: square, WIDS: diamond), and color denotes aggregation process (FedAvg: blue, FedProx: orange, Scaffold: green). The same mappings apply in both figures.


In [14]:
from functools import lru_cache


def csv_rows(file):
    # Parse one label/index column only; never load the feature matrix.
    return sum(len(chunk) for chunk in pd.read_csv(file, usecols=[0], chunksize=100_000))


@lru_cache(maxsize=None)
def dataset_sizes(dataset, partition, parameter, seed):
    directory = partition_data_directory(dataset, partition, parameter, seed)
    ids = partition_client_ids(directory)
    train_files = [directory / f"client_{i}_y_training.csv" for i in ids]
    test_files = [directory / f"client_{i}_y_test.csv" for i in ids]
    global_test = directory / "global_y_test.csv"
    train = sum(csv_rows(file) for file in train_files) if train_files else np.nan
    test = sum(csv_rows(file) for file in test_files) if test_files and all(
        file.is_file() for file in test_files) else np.nan
    return {"Training_samples": train, "Client_test_samples": test,
            "Global_test_samples": csv_rows(global_test) if global_test.is_file() else np.nan}


size_records = []
for _, experiment in experiment_times.iterrows():
    values = dataset_sizes(experiment.Dataset, experiment.Type_of_partition,
                           str(experiment["Dirichlet/NamePartition"]), int(experiment.Random_Seed))
    size_records.append({**experiment.to_dict(), **values})
scalability_data = pd.DataFrame(size_records)
if scalability_data.empty:
    print("No retained experiments are available for scalability analysis.")
else:
    scalability_summary = scalability_data.groupby(
        ["Dataset", "Aggregation_Process", "Clients"], dropna=False).agg(
        Experiments=("Random_Seed", "size"),
        Total_rounds=("Rounds", "sum"),
        Rounds_per_experiment_min=("Rounds", "min"),
        Rounds_per_experiment_mean=("Rounds", "mean"),
        Rounds_per_experiment_max=("Rounds", "max"),
        Training_samples_min=("Training_samples", "min"),
        Training_samples_max=("Training_samples", "max"),
        Client_test_samples_min=("Client_test_samples", "min"),
        Client_test_samples_max=("Client_test_samples", "max"),
        Server_SV_total_mean_s=("Server_SV_s", "mean"),
        Client_SV_total_mean_s=("Client_SV_s", "mean"),
        Combined_SV_total_mean_s=("Combined_SV_s", "mean"),
        Combined_SV_total_std_s=("Combined_SV_s", "std"))
    display(scalability_summary)
    scalability_figure = make_subplots(rows=1, cols=3, horizontal_spacing=0.10,
        subplot_titles=["SV time vs client count", "SV time vs round count", "SV time vs test-set size"])
    # Shared visual encoding for the round-count and test-size figures.
    aggregation_colors = {"FedAvg": "#1f77b4", "FedProx": "#ff7f0e", "Scaffold": "#2ca02c"}
    dataset_symbols = {"har": "circle", "edge-iot-coreset": "square", "wids_dataset": "diamond"}
    for index, ((dataset, strategy), group) in enumerate(scalability_data.groupby(
            ["Dataset", "Aggregation_Process"])):
        label = f"{dataset} / {strategy}"
        color = qualitative.Plotly[index % len(qualitative.Plotly)]
        for col, x_column in [(1, "Clients"), (2, "Rounds"), (3, "Client_test_samples")]:
            marker = dict(color=color, size=8, opacity=0.65)
            if col in (2, 3):
                marker.update(color=aggregation_colors[strategy], symbol=dataset_symbols[dataset], size=9)
            scalability_figure.add_trace(go.Scatter(x=group[x_column], y=group.Combined_SV_s,
                mode="markers", name=label, legendgroup=label, showlegend=col == 1,
                marker=marker,
                customdata=group[["Random_Seed", "Rounds"]],
                hovertemplate="x=%{x}<br>Total SV=%{y:.3f} s/experiment<br>Seed=%{customdata[0]}"
                              "<br>Rounds=%{customdata[1]}<extra>%{fullData.name}</extra>"), row=1, col=col)
    for col, title in [(1, "Number of clients"), (2, "Analysed rounds per experiment"),
                       (3, "Total client test samples")]:
        scalability_figure.update_xaxes(title_text=title, row=1, col=col)
        scalability_figure.update_yaxes(title_text="Total combined SV seconds per experiment", row=1, col=col)
    scalability_figure.update_xaxes(tickvals=sorted(scalability_data.Clients.unique()), row=1, col=1)
    scalability_figure.update_layout(template="plotly_white",
        title=f"Total SV summed over all analysed rounds; {len(scalability_data)} experiments, "
              f"{int(scalability_data.Rounds.sum())} training rounds",
        legend=dict(orientation="h", yanchor="top", y=-0.22, x=0),
        margin=dict(t=90, b=170, l=70, r=30))
    total_sv_clients_figure = save_single_panel(scalability_figure, 1, "sv_total_vs_clients.pdf")
    total_sv_rounds_figure = save_single_panel(scalability_figure, 2, "sv_total_vs_rounds.pdf")
    total_sv_test_size_figure = save_single_panel(scalability_figure, 3, "sv_total_vs_test_size.pdf")


Experiments  Total_rounds  \
Dataset          Aggregation_Process Clients                              
edge-iot-coreset FedAvg              5                 35          1137   
                 FedProx             5                 31           670   
                 Scaffold            5                 35          1090   
har              FedAvg              6                 35          1055   
                 FedProx             6                 35          1223   
                 Scaffold            6                 35          1200   
wids_dataset     FedAvg              6                 30          1197   
                 FedProx             6                 30          1054   
                 Scaffold            6                 30          1145   

                                              Rounds_per_experiment_min  \
Dataset          Aggregation_Process Clients                              
edge-iot-coreset FedAvg              5                                2   
                 FedProx             5                                2   
                 Scaffold            5                                2   
har              FedAvg              6                               10   
                 FedProx             6                               10   
                 Scaffold            6                               16   
wids_dataset     FedAvg              6                                4   
                 FedProx             6                                5   
                 Scaffold            6                                2   

                                              Rounds_per_experiment_mean  \
Dataset          Aggregation_Process Clients                               
edge-iot-coreset FedAvg              5                         32.485714   
                 FedProx             5                         21.612903   
                 Scaffold            5                         31.142857   
har              FedAvg              6                         30.142857   
                 FedProx             6                         34.942857   
                 Scaffold            6                         34.285714   
wids_dataset     FedAvg              6                         39.900000   
                 FedProx             6                         35.133333   
                 Scaffold            6                         38.166667   

                                              Rounds_per_experiment_max  \
Dataset          Aggregation_Process Clients                              
edge-iot-coreset FedAvg              5                               50   
                 FedProx             5                               50   
                 Scaffold            5                               50   
har              FedAvg              6                               50   
                 FedProx             6                               50   
                 Scaffold            6                               50   
wids_dataset     FedAvg              6                               50   
                 FedProx             6                               50   
                 Scaffold            6                               50   

                                              Training_samples_min  \
Dataset          Aggregation_Process Clients                         
edge-iot-coreset FedAvg              5                      334160   
                 FedProx             5                      334160   
                 Scaffold            5                      334160   
har              FedAvg              6                        6006   
                 FedProx             6                        6006   
                 Scaffold            6                        6006   
wids_dataset     FedAvg              6                       53060   
                 FedProx             6                       53060   
                 Scaffold      

Saved ../../notebooks/NotebooksActualExperimentation/images/sv_total_vs_clients.pdf


Saved ../../notebooks/NotebooksActualExperimentation/images/sv_total_vs_rounds.pdf


Saved ../../notebooks/NotebooksActualExperimentation/images/sv_total_vs_test_size.pdf


## Mean and standard deviation of total SV time per experiment

For every experiment, combined server + client SV intervals are summed over all its analysed rounds. The third figure shows the mean ± one sample standard deviation of these **experiment totals**, grouped by dataset and by client test-set size within each dataset. Each experiment receives equal weight. Strategies, partitions, and round counts are pooled in these two panels, so all contribute to their variation. This is total execution cost, not a comparison normalized to equal training lengths.

`configuration_total_sv_summary` also reports means and standard deviations across seeds within each training configuration (the first four experiment columns). Tables report experiment count, total analysed rounds, and minimum/mean/maximum rounds per experiment. A group with one experiment has undefined sample SD, displayed as no error-bar width.


In [15]:
if scalability_data.empty:
    print("No retained experiments with dataset-size metadata are available.")
else:
    total_sv_aggregations = dict(
        Experiments=("Combined_SV_s", "size"), Total_rounds=("Rounds", "sum"),
        Rounds_min=("Rounds", "min"), Rounds_mean=("Rounds", "mean"),
        Rounds_max=("Rounds", "max"),
        SV_total_mean_s=("Combined_SV_s", "mean"), SV_total_std_s=("Combined_SV_s", "std"))
    configuration_total_sv_summary = scalability_data.groupby(
        CONFIG_COLUMNS, dropna=False).agg(**total_sv_aggregations)
    sv_total_summary_by_dataset = scalability_data.groupby("Dataset").agg(**total_sv_aggregations)
    sv_total_summary_by_test_size = scalability_data.groupby(
        ["Dataset", "Client_test_samples"], dropna=True).agg(**total_sv_aggregations)
    display(configuration_total_sv_summary)
    display(sv_total_summary_by_dataset)
    display(sv_total_summary_by_test_size)
    total_variability_figure = make_subplots(rows=1, cols=2, horizontal_spacing=0.13,
        subplot_titles=["Total SV per experiment by dataset", "Total SV per experiment by test-set size"])
    dataset_colors = {dataset: qualitative.Plotly[index % len(qualitative.Plotly)]
                      for index, dataset in enumerate(sv_total_summary_by_dataset.index)}
    total_variability_figure.add_trace(go.Bar(x=sv_total_summary_by_dataset.index,
        y=sv_total_summary_by_dataset.SV_total_mean_s, name="Mean ± SD", showlegend=False,
        marker_color=[dataset_colors[dataset] for dataset in sv_total_summary_by_dataset.index],
        error_y=dict(type="data", array=sv_total_summary_by_dataset.SV_total_std_s.fillna(0)),
        customdata=sv_total_summary_by_dataset[["Experiments", "Rounds_mean"]],
        hovertemplate="%{x}<br>Mean total SV=%{y:.3f} s/experiment"
                      "<br>Experiments=%{customdata[0]}<br>Mean rounds=%{customdata[1]:.1f}<extra></extra>"),
        row=1, col=1)
    for dataset, group in sv_total_summary_by_test_size.reset_index().groupby("Dataset"):
        group = group.sort_values("Client_test_samples")
        total_variability_figure.add_trace(go.Scatter(x=group.Client_test_samples, y=group.SV_total_mean_s,
            mode="markers", name=dataset, marker=dict(color=dataset_colors[dataset], size=10),
            error_y=dict(type="data", array=group.SV_total_std_s.fillna(0)),
            customdata=group[["Total_rounds", "Experiments", "Rounds_mean"]],
            hovertemplate="Test samples=%{x}<br>Mean total SV=%{y:.3f} s/experiment"
                          "<br>Total rounds=%{customdata[0]}<br>Experiments=%{customdata[1]}"
                          "<br>Mean rounds=%{customdata[2]:.1f}<extra>%{fullData.name}</extra>"), row=1, col=2)
    total_variability_figure.update_xaxes(title_text="Dataset", row=1, col=1)
    total_variability_figure.update_xaxes(title_text="Total client test samples", row=1, col=2)
    total_variability_figure.update_yaxes(title_text="Total combined SV seconds per experiment")
    total_variability_figure.update_layout(template="plotly_white",
        title=f"Totals over each experiment's analysed rounds; {len(scalability_data)} experiments, "
              f"{int(scalability_data.Rounds.sum())} training rounds",
        legend=dict(orientation="h", yanchor="top", y=-0.18, x=0.55),
        margin=dict(t=90, b=130, l=90, r=30))
    total_sv_dataset_summary_figure = save_single_panel(
        total_variability_figure, 1, "sv_total_mean_std_by_dataset.pdf")
    total_sv_test_size_summary_figure = save_single_panel(
        total_variability_figure, 2, "sv_total_mean_std_by_test_size.pdf")


Experiments  \
Aggregation_Process Dataset          Type_of_partition Dirichlet/NamePartition                                           
FedAvg              edge-iot-coreset dirichlet         0.1                                                           5   
                                                       10                                                            5   
                                                       1000                                                          5   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                        5   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                        5   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                           5   
                                                       edgeiot_coreset_1_Maverick_sql_injection                      5   
                    har              dirichlet         0.1                                                           5   
                                                       10                                                            5   
                                                       1000                                                          5   
                                     manual            HAR_1_Maverick_Laying                                         5   
                                                       HAR_1_Maverick_Laying_Balanced                                5   
                                                       HAR_1_Maverick_WalkingUpstairs                                5   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced                       5   
                    wids_dataset     manual            wids_energy_feature_skew_building_type                        5   
                                                       wids_energy_feature_skew_state_factor                         5   
                                                       wids_energy_iid_sampling                                      5   
                                                       wids_energy_maverick_facility_type_grocery_store              5   
                                                       wids_energy_maverick_facility_type_uncategorize...            5   
                                                       wids_energy_non_iid_by_label                                  5   
FedProx             edge-iot-coreset dirichlet         0.1                                                           5   
                                                       10                                                            5   
                                                       1000                                                          5   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                        1   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                        5   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                           5   
                                                       edgeiot_coreset_1_Maverick_sql_injection                      5   
                    har              dirichlet         0.1                                                           5   
                                                       10                                                            5   
                                                       1000                                                          5   
                                     manual            HAR_1_Maverick_Laying                                         5   
                                                       HAR_1_Maverick_Laying_Bala

,Experiments,Total_rounds,Rounds_min,Rounds_mean,Rounds_max,SV_total_mean_s,SV_total_std_s
Dataset,,,,,,,
edge-iot-coreset,101,2897,2,28.683168,50,930.412293,608.661344
har,105,3478,10,33.123810,50,46.981214,15.944009
wids_dataset,90,3396,2,37.733333,50,347.314241,142.761480


Experiments  Total_rounds  Rounds_min  \
Dataset          Client_test_samples                                          
edge-iot-coreset 143192                        26          1108           4   
                 143193                        30           497           2   
                 143226                        45          1292           2   
har              2417                          15           601          18   
                 2472                          15           444          19   
                 2824                           3            90          16   
                 2835                           3            82          10   
                 2836                           6           130          11   
                 2841                           3            85          22   
                 2842                           6           133          18   
                 2845                           3            78          14   
                 2846                           6           292          42   
                 2847                           3           102          10   
                 2849                           3           128          37   
                 2855                           3           120          34   
                 2859                           3            89          24   
                 2868                           3            85          23   
                 2931                          30          1019          10   
wids_dataset     22697                         90          3396           2   

                                      Rounds_mean  Rounds_max  \
Dataset          Client_test_samples                            
edge-iot-coreset 143192                 42.615385          50   
                 143193                 16.566667          39   
                 143226                 28.711111          50   
har              2417                   40.066667          50   
                 2472                   29.600000          39   
                 2824                   30.000000          45   
                 2835                   27.333333          37   
                 2836                   21.666667          32   
                 2841                   28.333333          32   
                 2842                   22.166667          25   
                 2845                   26.000000          41   
                 2846                   48.666667          50   
                 2847                   34.000000          50   
                 2849                   42.666667          50   
                 2855                   40.000000          50   
                 2859                   29.666667          36   
                 2868                   28.333333          35   
                 2931                   33.966667          50   
wids_dataset     22697                  37.733333          50   

                                      SV_total_mean_s  SV_total_std_s  
Dataset          Client_test_samples                                   
edge-iot-coreset 143192                   1333.232232      510.183450  
                 143193                    655.109762      511.339993  
                 143226                    881.206905      609.685773  
har              2417                       49.842620       11.768274  
                 2472                       39.952874        9.374556  
                 2824                       40.683315       20.368979  
                 2835                       35.783602       18.331710  
                 2836                       31.481900       10.014998  
                 2841                       40.642662        9.329189  
                 2842                       30.430626        4.432075  
                 2845                       37.697897       19.714224  
                 2846                       64.337025        9.135590  
    

Saved ../../notebooks/NotebooksActualExperimentation/images/sv_total_mean_std_by_dataset.pdf


Saved ../../notebooks/NotebooksActualExperimentation/images/sv_total_mean_std_by_test_size.pdf
